# BMW Used Cars — Exploratory Data Analysis

**Author:** Alan Omelchenco  
**Tools:** Python · pandas · matplotlib · seaborn  
**Dataset:** `bmw_used_cars.csv` — 10,781 records sourced from Kaggle  
**Source:** [BMW Used Car Listing — Kaggle](https://www.kaggle.com/datasets/mysarahmadbhat/bmw-used-car-listing)

---

## Objectives

This project performs an exploratory data analysis (EDA) on a dataset of used BMW vehicles sold in the UK market. The goal is to understand the factors that influence the resale price of these vehicles and extract actionable insights about the used car market.

Specific objectives:

1. Understand the structure and quality of the dataset — data types, null values, distributions and outliers.
2. Clean and prepare the data for analysis — handle inconsistencies and create derived variables where needed.
3. Analyze price distribution across models, fuel types and transmission types.
4. Examine how mileage and vehicle age affect resale price.
5. Identify which models retain value best over time.
6. Explore efficiency metrics (mpg) across fuel types and engine sizes.
7. Summarize key findings and business insights.

---


## 0. Libraries

We start by importing all the libraries needed for the analysis. Each one serves a specific purpose:

- **pandas**: loading, cleaning and transforming tabular data.
- **numpy**: numerical operations and statistical calculations.
- **matplotlib / seaborn**: building clear and informative visualizations.
- **warnings**: suppressing minor alerts that don't affect the analysis.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Global style settings applied to all charts in this notebook
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.color":       "#e5e5e5",
    "grid.linewidth":   0.7,
    "font.family":      "DejaVu Sans",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
})

# Color palette used consistently throughout the notebook
DARK_BLUE  = "#1F3A5F"
MID_BLUE   = "#2E75B6"
LIGHT_BLUE = "#9DC3E6"
RED        = "#C0392B"
GREEN      = "#1E7145"
GRAY       = "#666666"
PALETA     = [DARK_BLUE, MID_BLUE, LIGHT_BLUE, "#F0A500", RED, GREEN, "#7B5EA7", "#2E8B57"]

OUTPUT = "../img"

print("Libraries imported successfully.")

## 1. Loading the dataset

We load the CSV file using `pandas`. We also strip whitespace from column names and string values, since some entries in this dataset contain leading spaces that could cause grouping errors.


In [ ]:
# Load the dataset from the data folder
df = pd.read_csv("../data/bmw_used_cars.csv")

# Strip leading/trailing whitespace from column names and string columns
# This is a common issue with CSV exports and can cause silent grouping errors
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns.")
df.head()

## 2. Initial exploration

Before cleaning or analyzing anything, we inspect the raw dataset to understand what we're working with: data types, value ranges, cardinality of categorical variables and the presence of null values or outliers.


In [ ]:
# Data types and non-null counts for each column
print("Column types and non-null counts:")
df.info()

In [ ]:
# Descriptive statistics for all numeric columns
# Pay attention to min/max values — extreme values may indicate data entry errors
df.describe().round(1)

In [ ]:
# Check for null values across all columns
nulls = df.isnull().sum()
print("Null values per column:")
print(nulls[nulls > 0] if nulls.sum() > 0 else "No null values found.")

In [ ]:
# Cardinality of categorical variables
# This tells us how many unique values each category has
print(f"Models      : {df['model'].nunique()} unique — {sorted(df['model'].unique())}")
print(f"Transmission: {df['transmission'].nunique()} unique — {df['transmission'].unique().tolist()}")
print(f"Fuel type   : {df['fuelType'].nunique()} unique — {df['fuelType'].unique().tolist()}")
print(f"Year range  : {df['year'].min()} to {df['year'].max()}")

In [ ]:
# Checking for duplicate rows — exact copies across all columns
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")
if duplicates == 0:
    print("No duplicates found.")

## 3. Data cleaning and preparation

The dataset is largely clean, but we need to address a few issues before the analysis:

1. **Outliers in price**: the max price is £123,456, which may be a placeholder or data entry error. We'll investigate and filter if needed.
2. **Engine size = 0.0**: some rows have an engine size of 0, which is not physically possible.
3. **Extreme mpg values**: the max is 470.8 mpg, which is unrealistic even for hybrid vehicles.
4. **Derived variables**: we'll create `age` (years since registration) and `price_per_year` to support further analysis.


In [ ]:
# ── Investigating suspicious values ─────────────────────────────────────────

# Rows where engine size is 0 (physically impossible for combustion engines)
zero_engine = df[df["engineSize"] == 0]
print(f"Rows with engineSize = 0: {len(zero_engine)}")
print(zero_engine["fuelType"].value_counts())
# Note: electric vehicles can legitimately have engineSize = 0 in this dataset

In [ ]:
# Rows with extreme mpg values (above 150 mpg is unrealistic)
extreme_mpg = df[df["mpg"] > 150]
print(f"Rows with mpg > 150: {len(extreme_mpg)}")
print(extreme_mpg[["model", "fuelType", "mpg", "engineSize"]].head(10))

In [ ]:
# ── Applying filters ─────────────────────────────────────────────────────────

CURRENT_YEAR = 2020  # Most recent year in the dataset

df_clean = df.copy()

# Remove rows with mpg > 150 — these are likely data entry errors
# (except for Electric vehicles which don't use mpg as a meaningful metric)
df_clean = df_clean[~((df_clean["mpg"] > 150) & (df_clean["fuelType"] != "Electric"))]

# Remove rows where engineSize is 0 and fuel type is not Electric
# A combustion or hybrid engine must have a positive engine size
df_clean = df_clean[~((df_clean["engineSize"] == 0) & (df_clean["fuelType"] != "Electric"))]

# Remove the single anomalous price (123,456 — likely a placeholder)
df_clean = df_clean[df_clean["price"] < 120000]

print(f"Rows before cleaning : {len(df):,}")
print(f"Rows after cleaning  : {len(df_clean):,}")
print(f"Rows removed         : {len(df) - len(df_clean):,}")

In [ ]:
# ── Creating derived variables ────────────────────────────────────────────────

# 'age': how old is the car at the time of listing (relative to 2020)
df_clean["age"] = CURRENT_YEAR - df_clean["year"]

# 'age_group': categorical grouping for age-based analysis
bins   = [0, 2, 5, 10, 15, 100]
labels = ["0-2 yrs", "3-5 yrs", "6-10 yrs", "11-15 yrs", "15+ yrs"]
df_clean["age_group"] = pd.cut(df_clean["age"], bins=bins, labels=labels, right=True)

# 'mileage_band': grouping mileage into ranges for visual analysis
mbins   = [0, 10000, 30000, 60000, 100000, 300000]
mlabels = ["<10k", "10-30k", "30-60k", "60-100k", "100k+"]
df_clean["mileage_band"] = pd.cut(df_clean["mileage"], bins=mbins, labels=mlabels, right=True)

print("Derived variables created:")
print(f"  age        — range: {df_clean['age'].min()} to {df_clean['age'].max()} years")
print(f"  age_group  — {df_clean['age_group'].value_counts().to_dict()}")
print(f"  mileage_band — {df_clean['mileage_band'].value_counts().sort_index().to_dict()}")

## 4. Price distribution

We start the analysis by understanding how prices are distributed across the dataset. A right-skewed distribution is typical in car markets — most cars cluster in a mid-range, with fewer high-end vehicles pulling the mean upward.


In [ ]:
# ── Overall price distribution ────────────────────────────────────────────────
# A histogram shows the shape of the distribution.
# The KDE (kernel density estimate) overlaid as a line smooths it out.
# We also mark the median and mean to show the direction of skew.

fig, ax = plt.subplots(figsize=(11, 5))

ax.hist(df_clean["price"], bins=60, color=DARK_BLUE, edgecolor="white", alpha=0.85)

# Add vertical lines for mean and median
mean_price   = df_clean["price"].mean()
median_price = df_clean["price"].median()
ax.axvline(mean_price,   color=RED,   linestyle="--", linewidth=1.8, label=f"Mean:   £{mean_price:,.0f}")
ax.axvline(median_price, color=GREEN, linestyle="--", linewidth=1.8, label=f"Median: £{median_price:,.0f}")

ax.set_title("Price Distribution — BMW Used Cars", fontweight="bold", pad=14)
ax.set_xlabel("Price (£)")
ax.set_ylabel("Number of listings")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"£{v:,.0f}"))
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(f"{OUTPUT}/01_price_distribution.png", dpi=150)
plt.show()
print("Chart saved.")

In [ ]:
# ── Price by model (median) ───────────────────────────────────────────────────
# We use median instead of mean here because it's more robust to extreme values.
# Models are sorted by median price to make the chart easier to read.

price_by_model = (
    df_clean.groupby("model")["price"]
    .median()
    .sort_values()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 7))

bars = ax.barh(price_by_model["model"], price_by_model["price"],
               color=DARK_BLUE, edgecolor="white")

# Add value label at the end of each bar
for bar, val in zip(bars, price_by_model["price"]):
    ax.text(val + 300, bar.get_y() + bar.get_height() / 2,
            f"£{val:,.0f}", va="center", fontsize=8.5, color=DARK_BLUE)

ax.set_title("Median Resale Price by Model", fontweight="bold", pad=14)
ax.set_xlabel("Median Price (£)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"£{v:,.0f}"))

plt.tight_layout()
plt.savefig(f"{OUTPUT}/02_price_by_model.png", dpi=150)
plt.show()
print("Chart saved.")

**Key observation:** M-series and larger models (7 Series, 8 Series) command significantly higher resale prices, while the 1 Series and i3 sit at the lower end of the range. This reflects both original purchase price and brand positioning within BMW's lineup.


## 5. Impact of mileage and age on price

Two of the strongest predictors of resale value in any used car market are mileage and vehicle age. Here we examine both relationships to understand how quickly BMW vehicles depreciate.


In [ ]:
# ── Scatter plot: price vs mileage ────────────────────────────────────────────
# Each point is a car listing. We color by fuel type to check whether
# diesel/petrol vehicles follow different depreciation patterns.
# Alpha is set low (0.25) to handle the high density of points.

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: price vs mileage
fuel_colors = {"Diesel": DARK_BLUE, "Petrol": RED, "Hybrid": GREEN,
               "Electric": "#F0A500", "Other": GRAY}

for fuel, grp in df_clean.groupby("fuelType"):
    ax1.scatter(grp["mileage"], grp["price"], alpha=0.18, s=8,
                color=fuel_colors.get(fuel, GRAY), label=fuel)

ax1.set_title("Price vs Mileage", fontweight="bold")
ax1.set_xlabel("Mileage (miles)")
ax1.set_ylabel("Price (£)")
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1000:.0f}k"))
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"£{v/1000:.0f}k"))
ax1.legend(fontsize=8, markerscale=3)

# Right: median price by age group
# We aggregate by age_group to show the average depreciation curve
age_price = (
    df_clean.groupby("age_group", observed=True)["price"]
    .median()
    .reset_index()
)

ax2.bar(age_price["age_group"].astype(str), age_price["price"],
        color=PALETA[:len(age_price)], edgecolor="white")

for i, (_, row) in enumerate(age_price.iterrows()):
    ax2.text(i, row["price"] + 200, f"£{row['price']:,.0f}",
             ha="center", fontsize=9, color=DARK_BLUE)

ax2.set_title("Median Price by Vehicle Age", fontweight="bold")
ax2.set_xlabel("Age group")
ax2.set_ylabel("Median Price (£)")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"£{v:,.0f}"))

plt.suptitle("Depreciation Analysis — Mileage and Age", fontweight="bold", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f"{OUTPUT}/03_depreciation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved.")

**Key observation:** The expected negative relationship between mileage and price is clearly visible. Vehicles aged 0–2 years retain the highest value, with a steep depreciation cliff in the first 5 years. After 10 years, the rate of depreciation slows considerably.


## 6. Price by transmission and fuel type

Transmission type and fuel type both reflect buyer preferences and market demand. Automatic and semi-automatic vehicles generally command higher prices in the used market, and diesel vehicles have historically been preferred in the UK for long-distance driving.


In [ ]:
# ── Box plots: price by transmission and fuel type ────────────────────────────
# Box plots show the full distribution (median, IQR, range) rather than just
# an average, which makes them well-suited for comparing groups with price data.
# We filter out 'Other' fuel type as it has very few entries.

df_plot = df_clean[df_clean["fuelType"] != "Other"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Transmission
trans_order = df_plot.groupby("transmission")["price"].median().sort_values(ascending=False).index
sns.boxplot(data=df_plot, x="transmission", y="price",
            order=trans_order, palette=[DARK_BLUE, MID_BLUE, LIGHT_BLUE],
            width=0.5, flierprops=dict(marker=".", markersize=2, alpha=0.3), ax=ax1)
ax1.set_title("Price by Transmission Type", fontweight="bold")
ax1.set_xlabel("")
ax1.set_ylabel("Price (£)")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"£{v:,.0f}"))

# Fuel type
fuel_order = df_plot.groupby("fuelType")["price"].median().sort_values(ascending=False).index
sns.boxplot(data=df_plot, x="fuelType", y="price",
            order=fuel_order, palette=PALETA[:len(fuel_order)],
            width=0.5, flierprops=dict(marker=".", markersize=2, alpha=0.3), ax=ax2)
ax2.set_title("Price by Fuel Type", fontweight="bold")
ax2.set_xlabel("")
ax2.set_ylabel("Price (£)")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"£{v:,.0f}"))

plt.suptitle("Price Distribution by Transmission and Fuel Type", fontweight="bold", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f"{OUTPUT}/04_price_transmission_fuel.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved.")

**Key observation:** Automatic and semi-automatic vehicles carry a clear price premium over manual ones. Electric and hybrid vehicles show the widest price range, reflecting the diversity of models in those categories and the relatively recent market for EVs.


## 7. Value retention by model

One of the most practically useful analyses for a buyer or seller is understanding which models hold their value best over time. We calculate the average price drop per year of age for each model, which gives a comparable depreciation rate across models with different base prices.


In [ ]:
# ── Value retention: average price by model and age group ─────────────────────
# We filter to the most common age groups (0-10 years) where we have enough
# data points per model to make the comparison reliable.

# Models with enough data across age groups
common_models = ["1 Series", "2 Series", "3 Series", "4 Series",
                 "5 Series", "X1", "X3", "X5"]

df_retention = df_clean[
    (df_clean["model"].isin(common_models)) &
    (df_clean["age_group"].isin(["0-2 yrs", "3-5 yrs", "6-10 yrs"]))
]

retention_pivot = df_retention.pivot_table(
    values="price",
    index="model",
    columns="age_group",
    aggfunc="median",
    observed=True
)

# Calculate retention ratio: price at 6-10 yrs relative to price at 0-2 yrs
# Higher ratio = better value retention
retention_pivot["retention_ratio"] = (
    retention_pivot["6-10 yrs"] / retention_pivot["0-2 yrs"] * 100
).round(1)

retention_sorted = retention_pivot.sort_values("retention_ratio", ascending=True)

print("Value retention (median price at 6-10 yrs as % of price at 0-2 yrs):")
print(retention_sorted[["0-2 yrs", "3-5 yrs", "6-10 yrs", "retention_ratio"]].to_string())

In [ ]:
# ── Visualization: retention ratio by model ───────────────────────────────────
# A horizontal bar chart sorted by retention ratio.
# Color encodes retention level: green = good retention, red = faster depreciation.

fig, ax = plt.subplots(figsize=(10, 5))

colors = [GREEN if v >= 40 else MID_BLUE if v >= 30 else RED
          for v in retention_sorted["retention_ratio"]]

bars = ax.barh(retention_sorted.index, retention_sorted["retention_ratio"],
               color=colors, edgecolor="white")

for bar, val in zip(bars, retention_sorted["retention_ratio"]):
    ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", fontsize=9)

# Reference line at 35% as a benchmark
ax.axvline(35, color=GRAY, linestyle="--", linewidth=1.4, label="35% benchmark")
ax.set_title("Value Retention by Model
(Median price at 6-10 yrs as % of 0-2 yrs)", fontweight="bold", pad=14)
ax.set_xlabel("Retention ratio (%)")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUTPUT}/05_value_retention.png", dpi=150)
plt.show()
print("Chart saved.")

**Key observation:** Value retention varies significantly across models. SUV models (X3, X5) tend to hold their value better than saloon equivalents, reflecting sustained demand in the used market. The 1 Series depreciates fastest, consistent with its positioning as an entry-level model.


## 8. Fuel efficiency analysis

We examine the relationship between engine size and fuel efficiency (mpg), broken down by fuel type. This helps identify which combinations offer the best balance between performance and running costs.


In [ ]:
# ── MPG by fuel type and engine size ─────────────────────────────────────────
# We exclude Electric vehicles from the mpg analysis since mpg is not a
# meaningful metric for EVs — their efficiency is measured in miles per kWh.
# We also exclude engine size 0 and mpg outliers already removed in cleaning.

df_eff = df_clean[
    (df_clean["fuelType"].isin(["Diesel", "Petrol", "Hybrid"])) &
    (df_clean["engineSize"] > 0)
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: average mpg by fuel type
mpg_fuel = df_eff.groupby("fuelType")["mpg"].median().sort_values(ascending=False)
bars = ax1.bar(mpg_fuel.index, mpg_fuel.values,
               color=[DARK_BLUE, MID_BLUE, GREEN], edgecolor="white", width=0.5)

for bar, val in zip(bars, mpg_fuel.values):
    ax1.text(bar.get_x() + bar.get_width() / 2, val + 0.5,
             f"{val:.1f}", ha="center", fontsize=10, fontweight="bold")

ax1.set_title("Median MPG by Fuel Type", fontweight="bold")
ax1.set_xlabel("")
ax1.set_ylabel("Miles per gallon (mpg)")

# Right: scatter of engine size vs mpg, colored by fuel type
for fuel, color in zip(["Diesel", "Petrol", "Hybrid"], [DARK_BLUE, RED, GREEN]):
    subset = df_eff[df_eff["fuelType"] == fuel]
    ax2.scatter(subset["engineSize"], subset["mpg"], alpha=0.2, s=10,
                color=color, label=fuel)

ax2.set_title("Engine Size vs MPG", fontweight="bold")
ax2.set_xlabel("Engine size (litres)")
ax2.set_ylabel("MPG")
ax2.legend(fontsize=9, markerscale=3)

plt.suptitle("Fuel Efficiency Analysis", fontweight="bold", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f"{OUTPUT}/06_efficiency.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved.")

**Key observation:** Diesel vehicles consistently achieve higher mpg figures than petrol equivalents, which explains their historic popularity in the UK used market. Larger engine sizes correlate with lower efficiency, as expected. Hybrid vehicles show a wide efficiency range depending on driving conditions reported.


## 9. Market composition

Before closing the analysis, we look at the overall composition of the dataset — how listings are distributed across models, transmission types and fuel types. This reflects the supply side of the used BMW market.


In [ ]:
# ── Market composition: model, transmission, fuel type ───────────────────────
# We use count-based bar charts to show the volume of listings per category.
# This is useful context for interpreting the price analyses above —
# categories with few listings produce less reliable median estimates.

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Model distribution
model_counts = df_clean["model"].value_counts()
axes[0].barh(model_counts.index, model_counts.values, color=DARK_BLUE, edgecolor="white")
axes[0].set_title("Listings by Model", fontweight="bold")
axes[0].set_xlabel("Number of listings")

# Transmission distribution
trans_counts = df_clean["transmission"].value_counts()
axes[1].bar(trans_counts.index, trans_counts.values,
            color=[DARK_BLUE, MID_BLUE, LIGHT_BLUE], edgecolor="white", width=0.5)
for bar, val in zip(axes[1].patches, trans_counts.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, val + 50,
                 f"{val:,}", ha="center", fontsize=9)
axes[1].set_title("Listings by Transmission", fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("Number of listings")

# Fuel type distribution
fuel_counts = df_clean["fuelType"].value_counts()
axes[2].bar(fuel_counts.index, fuel_counts.values,
            color=PALETA[:len(fuel_counts)], edgecolor="white", width=0.5)
for bar, val in zip(axes[2].patches, fuel_counts.values):
    axes[2].text(bar.get_x() + bar.get_width() / 2, val + 30,
                 f"{val:,}", ha="center", fontsize=9)
axes[2].set_title("Listings by Fuel Type", fontweight="bold")
axes[2].set_xlabel("")

plt.suptitle("Market Composition", fontweight="bold", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f"{OUTPUT}/07_market_composition.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved.")

## 10. Summary and conclusions

A final summary of the key findings from the analysis.


In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────

most_listed  = df_clean["model"].value_counts().idxmax()
highest_med  = df_clean.groupby("model")["price"].median().idxmax()
lowest_med   = df_clean.groupby("model")["price"].median().idxmin()
best_mpg     = df_clean[df_clean["fuelType"] == "Diesel"]["mpg"].median()
pct_auto     = (df_clean["transmission"] == "Automatic").mean() * 100
pct_diesel   = (df_clean["fuelType"] == "Diesel").mean() * 100
avg_age      = df_clean["age"].mean()
avg_mileage  = df_clean["mileage"].mean()

print("=" * 58)
print("   SUMMARY — BMW USED CARS EDA")
print("=" * 58)
print(f"  Total listings analysed  : {len(df_clean):,}")
print(f"  Most listed model        : {most_listed}")
print(f"  Highest median price     : {highest_med}")
print(f"  Lowest median price      : {lowest_med}")
print(f"  % Automatic transmission : {pct_auto:.1f}%")
print(f"  % Diesel vehicles        : {pct_diesel:.1f}%")
print(f"  Average vehicle age      : {avg_age:.1f} years")
print(f"  Average mileage          : {avg_mileage:,.0f} miles")
print(f"  Median diesel MPG        : {best_mpg:.1f}")
print("=" * 58)
print()
print("Key findings:")
print("  1. Vehicle age and mileage are the strongest drivers of price.")
print("  2. M-series and 7/8 Series command the highest resale values.")
print("  3. SUVs (X3, X5) retain value better than saloon equivalents.")
print("  4. Automatic transmission carries a consistent price premium.")
print("  5. Diesel dominates the used market but hybrid/EV share is growing.")
print("  6. Depreciation is steepest in the first 5 years of ownership.")

---

## Files generated

| File | Description |
|---|---|
| `img/01_price_distribution.png` | Overall price distribution with mean and median |
| `img/02_price_by_model.png` | Median resale price ranked by model |
| `img/03_depreciation.png` | Price vs mileage and median price by age group |
| `img/04_price_transmission_fuel.png` | Price distribution by transmission and fuel type |
| `img/05_value_retention.png` | Value retention ratio by model (6-10 yrs vs 0-2 yrs) |
| `img/06_efficiency.png` | MPG by fuel type and engine size |
| `img/07_market_composition.png` | Volume of listings by model, transmission and fuel type |

---

**Author:** Alan Omelchenco · [LinkedIn](https://linkedin.com/in/alanomelchenco) · alannomelchenco@gmail.com
